# Additional End of week Exercise - week 3
Let's imagine this is a AI that lives in the CI/CD pipeline of a company and generates synthetic data for the test deployments of each PR pointing to main.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# ensures you have llama pulled
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [3]:
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

ai_models = [
    ("gpt-4o-mini", MODEL_GPT, "gpt-4o-mini"),
    ("llama3.2", MODEL_LLAMA, "ollama"),
]

In [4]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openAI = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

SYSTEM_PROMPT = """
You are the AI backend for synthetic data generator.
You will receive a prompt from the user.
You will generate a synthetic dataset based on the prompt.
You will return the dataset in JSON format.
You look at the prompt and decide which fields are relevant.
You reside in the CI/CD pipeline where you generate synthetic data to be loaded in the test DB.
"""

API key looks good so far


In [5]:
def stream_response(history, user_message, model_label):
    config_from_label = {label: (model_id, backend) for label, model_id, backend in ai_models}
    model_id, backend = config_from_label.get(model_label, ai_models[0][1:])

    client = openAI if backend == "gpt-4o-mini" else ollama
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant or ""})
    messages.append({"role": "user", "content": user_message})

    stream = client.chat.completions.create(
        model=model_id,
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        part = chunk.choices[0].delta.content or ""
        if part:
            yield part

In [6]:
def chat(message, history, model_choice):
    if not message or not message.strip():
        return
    full = ""
    for chunk in stream_response(history, message, model_choice):
        full += chunk
        yield full

In [ ]:
model_selector = gr.Dropdown(
    choices=[label for label, _, _ in ai_models],
    value=ai_models[0][0],
    label="Model",
)

ui = gr.ChatInterface(
    chat,
    additional_inputs=[model_selector],
    title="Synthetix",
    description="What data would you like to generate?",
)
ui.launch(share=True)